In [ ]:
import pandas as pd
import numpy as np
from sympy import Matrix
from IPython.display import display

In [2]:
# Motor Parmeters
rs  = 0.0;          # Stator resistance
rr  = 0.2826;       # Rotor resistance
ls  = 3.1364e-3;    # Stator inductance
lr  = 6.3264e-3;    # Rotor inductance
lm  = 109.9442e-3;  # Mutual inductance
j   = 0.192;        # Moment of inertia
npp = 2.0           # Number of pair poles

Ts = 100e-9

## Matrizes

In [5]:
K = 1.0/(lm*lm - ls*lr)

A = [
    [-Ts*rr/lr                 , -Ts*npp                    , Ts*lm*rr/lr                 , 0.0                          , 0.0],
    [Ts*npp                    , -Ts*rr/lr                  , 0.0                         , Ts*lm*rr/lr                  , 0.0],
    [-Ts*lm*rr*K/lr            , -Ts*lm*npp*K               , Ts*(lm*lm*rr*K/lr)+(lr*rs*K), 0.0                          , 0.0],
    [Ts*lm*npp*K               , -Ts*lm*rr*K/lr             , 0.0                         , Ts*(lm*lm*rr*K/lr)+(lr*rs*K) , 0.0],
    [Ts*(3.0*npp*lm)/(2.0*j*lr), Ts*(-3.0*npp*lm)/(2.0*j*lr), 0.0                         , 0.0                          , 0.0]
]

B = [
    [ 0.0     , 0.0     , 0.0],
    [ 0.0     , 0.0     , 0.0],
    [ -Ts*lr*K, 0.0     , 0.0],
    [ 0.0     , -Ts*lr*K, 0.0],
    [ 0.0     ,  0.0    , -Ts]
]

In [ ]:
pd.DataFrame(A)

,0,1,2,3,4
0,-4.466995e-06,-2.000000e-07,4.911202e-07,0.000000e+00,0.0
1,2.000000e-07,-4.466995e-06,0.000000e+00,4.911202e-07,0.0
2,-4.069646e-05,-1.822096e-06,4.474340e-06,0.000000e+00,0.0
3,1.822096e-06,-4.069646e-05,0.000000e+00,4.474340e-06,0.0
4,2.715412e-05,-2.715412e-05,0.000000e+00,0.000000e+00,0.0


In [14]:
pd.DataFrame(B)

,0,1,2
0,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00
2,-5.242344e-08,0.000000e+00,0.000000e+00
3,0.000000e+00,-5.242344e-08,0.000000e+00
4,0.000000e+00,0.000000e+00,-1.000000e-07


## Representação em Ponto Fixo

In [24]:
FP_INTEGER_BITS  = 14
FP_FRACTION_BITS = 28
TOTAL_BITS       = FP_INTEGER_BITS + FP_FRACTION_BITS  # 42 bits
SCALE            = 1 << FP_FRACTION_BITS                   # 2^28
MAX_INT          = (1 << (TOTAL_BITS - 1)) - 1             #  2^42 - 1
MIN_INT          = - (1 << (TOTAL_BITS - 1))               # -2^42
HEX_DIGITS       = (TOTAL_BITS + 3) // 4                   # ceil(43/4)=11 dígitos hex

def float_to_fixed(x: float) -> int:
    """Converte float para inteiro com escala (2's complement), com arredondamento e saturação."""
    scaled = int(round(x * SCALE))
    if scaled > MAX_INT:
        scaled = MAX_INT
    elif scaled < MIN_INT:
        scaled = MIN_INT
    return scaled

def fixed_to_twos_hex(v: int) -> str:
    """Converte inteiro assinado para hex de largura TOTAL_BITS (2's complement, zero-padded)."""
    mask = (1 << TOTAL_BITS) - 1
    twos = v & mask
    return f"0x{twos:0{HEX_DIGITS}X}"

def fixed_to_float(v: int) -> float:
    """Converte inteiro fixo (assinado) de volta pra float."""
    # detectar sinal considerando TOTAL_BITS
    if v >= (1 << (TOTAL_BITS - 1)):
        v = v - (1 << TOTAL_BITS)  # desfaz 2's complement
    return v / SCALE

In [25]:
def quantize_matrix(M):
    M_float = np.array(M, dtype=float)
    M_fixed = np.vectorize(float_to_fixed)(M_float)
    M_hex   = np.vectorize(fixed_to_twos_hex)(M_fixed)
    return M_float, M_fixed, M_hex

A_float, A_fixed, A_hex = quantize_matrix(A)
B_float, B_fixed, B_hex = quantize_matrix(B)

In [26]:
# === DataFrames bonitinhos ===
df_A_hex = pd.DataFrame(A_hex, columns=[f"x{j+1}" for j in range(A_hex.shape[1])],
                              index=[f"x{i+1}" for i in range(A_hex.shape[0])])
df_B_hex = pd.DataFrame(B_hex, columns=[f"u{j+1}" for j in range(B_hex.shape[1])],
                              index=[f"x{i+1}" for i in range(B_hex.shape[0])])

# === Mostrar em tabelas no notebook ===
display(pd.DataFrame({"Config": [f"Q{FP_INTEGER_BITS}.{FP_FRACTION_BITS}",
                                 f"TOTAL_BITS={TOTAL_BITS}",
                                 f"HEX_DIGITS={HEX_DIGITS}"]}))

print("Matriz A (hex, 2's complement, 42 bits):")
display(df_A_hex)

print("Matriz B (hex, 2's complement, 42 bits):")
display(df_B_hex)

,Config
0,Q14.28
1,TOTAL_BITS=42
2,HEX_DIGITS=11


Matriz A (hex, 2's complement, 42 bits):


,x1,x2,x3,x4,x5
x1,0x3FFFFFFFB51,0x3FFFFFFFFCA,0x00000000084,0x00000000000,0x00000000000
x2,0x00000000036,0x3FFFFFFFB51,0x00000000000,0x00000000084,0x00000000000
x3,0x3FFFFFFD554,0x3FFFFFFFE17,0x000000004B1,0x00000000000,0x00000000000
x4,0x000000001E9,0x3FFFFFFD554,0x00000000000,0x000000004B1,0x00000000000
x5,0x00000001C79,0x3FFFFFFE387,0x00000000000,0x00000000000,0x00000000000


Matriz B (hex, 2's complement, 42 bits):


,u1,u2,u3
x1,0x00000000000,0x00000000000,0x00000000000
x2,0x00000000000,0x00000000000,0x00000000000
x3,0x3FFFFFFFFF2,0x00000000000,0x00000000000
x4,0x00000000000,0x3FFFFFFFFF2,0x00000000000
x5,0x00000000000,0x00000000000,0x3FFFFFFFFE5
